## Working with timeseries

This notebook runs our ["Working with Timeseries"](https://docs.lsdb.io/en/latest/tutorials/pre_executed/timeseries.html) workflow with both Dask and Ray.

In [1]:
import lsdb
import numpy as np
from nested_pandas.utils import count_nested
from astropy.timeseries import LombScargle

We will compute the periodogram for ZTF DR22 objects with:

- perfectly clean extractions (lc.catflags == 0)
- r-band only (filterid == 2)
- median of lc.mag is brighter than 16
- at least 10 points

In [3]:
ztf_objects = lsdb.open_catalog(
    "https://data.lsdb.io/hats/ztf_dr22",
    search_filter=lsdb.ConeSearch(ra=254.5, dec=35.3, radius_arcsec=5*3600),
).nest_lists(
    list_columns=["hmjd", "mag", "magerr", "clrcoeff", "catflags"],
    name="lc",  # name to give the resulting nested column
)

filtered_catalog = ztf_objects.query("filterid == 2")
filtered_catalog = filtered_catalog.query("lc.catflags == 0")
# Using map_partitions to drop rows with NaN in lc.mag
filtered_catalog = filtered_catalog.map_partitions(lambda df: df.dropna(subset=["lc.mag"]))

def median_mag(row):
    return np.median(row["lc.mag"])

catalog_w_features = filtered_catalog.map_rows(
    median_mag,  # our user-defined function
    columns=["lc.mag"],  # names of the column(s) to pass to the function
    output_names=["median_mag"],  # name(s) of the new column(s)
    meta={"median_mag": float},  # for Dask: describe the type of the output
    append_columns=True,
)

catalog_w_features = catalog_w_features.query("median_mag < 16")
catalog_w_features = catalog_w_features.map_partitions(lambda df: count_nested(df, "lc"))
catalog_w_features = catalog_w_features.query("n_lc >= 10")

def extract_period(row):
    time = row["lc.hmjd"]
    mag = row["lc.mag"]
    error = row["lc.magerr"]
    ls = LombScargle(time, mag, error)
    freq, power = ls.autopower()
    argmax = np.argmax(power)
    period = 1.0 / freq[argmax]
    false_alarm_prob = ls.false_alarm_probability(power[argmax])
    return {"period": period, "false_alarm_prob": false_alarm_prob}

catalog_w_features = catalog_w_features.map_rows(
    extract_period,
    # Column names specifying function arguments
    columns=["lc.hmjd", "lc.mag", "lc.magerr"],
    # Returned data type shape
    meta={"period": float, "false_alarm_prob": float},
    append_columns=True,
)
periodic_catalog = catalog_w_features.query("false_alarm_prob < 1e-10 and 1.5 < period < 9.5")

periodic_catalog

,objectid,filterid,objra,objdec,nepochs,lc,median_mag,n_lc,period,false_alarm_prob
npartitions=39,,,,,,,,,,
"Order: 5, Pixel: 2325",int64[pyarrow],int8[pyarrow],float[pyarrow],float[pyarrow],int64[pyarrow],"nested<hmjd: [double], mag: [float], magerr: [...",float64,int32,float64,float64
"Order: 5, Pixel: 2326",...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...
"Order: 5, Pixel: 2406",...,...,...,...,...,...,...,...,...,...
"Order: 5, Pixel: 2408",...,...,...,...,...,...,...,...,...,...


This workflow includes calls to several core operations:
- `open_catalog`
- `map_partitions`
- `map_rows`
- `query`

### Compute with Dask

We're used to the Dask distributed Client and its parameters.

In [7]:
from dask.distributed import Client

In [ ]:
%%time
with Client(n_workers=4):
    dask_df = periodic_catalog.compute()

### Compute with Ray

Let's see how we can configure Ray's cluster similarly:

In [ ]:
import ray
from ray.util.dask import enable_dask_on_ray

In [ ]:
%%time
ray.init(num_cpus=4)
with enable_dask_on_ray():
    ray_df = periodic_catalog.compute()
ray.shutdown()

- Ray was significantly slower.
- Can we control the memory per worker?
- Need to see the ray dashboard for more info.
- Ray needs to convert the graph so that might be why.